### Load data

In [ ]:
import os
import sys

import zarr

sys.path.append("../../")

from PyriteUtility.computer_vision.imagecodecs_numcodecs import register_codecs

register_codecs()

if "PYRITE_DATASET_FOLDERS" not in os.environ:
    raise ValueError("Please set the environment variable PYRITE_DATASET_FOLDERS")
dataset_folder_path = os.environ.get("PYRITE_DATASET_FOLDERS")

dataset_path = dataset_folder_path + "/umi-ft/replay_test_2/"
id_list = [0]

print("dataset_path:", dataset_path)
buffer = zarr.open(dataset_path, mode="r")

### Exame data

In [ ]:
# # buffer.tree()
# for key in buffer['data'].keys():
#     print("buffer.keys:", key)

for key, value in buffer["data"].items():
    print("buffer.items:", key)

# ep = 'episode_0'
# data = buffer['data'][ep]

# for key in data.keys():
#     print("data keys: ", key)

# print(data["ts_pose_fb_0"].shape)

# Playback

### Launch the ManipServer

In [ ]:
import os
import sys
import time

import numpy as np

# from PyriteUtility.computer_vision.computer_vision_utility import get_image_transform
# from PyriteUtility.umi_utils.usb_util import reset_all_elgato_devices
from workcell.table_top_manip.python import (
    manip_server_pybind as ms,
)

input_res = (224, 224)
if "PYRITE_HARDWARE_CONFIG_FOLDERS" not in os.environ:
    raise ValueError(
        "Please set the environment variable PYRITE_HARDWARE_CONFIG_FOLDERS"
    )
config_folders = os.environ.get("PYRITE_HARDWARE_CONFIG_FOLDERS")

# reset_all_elgato_devices()

hardware_config_path = config_folders + "/right_arm_coinft.yaml"

Tr = np.eye(6)
n_af = 0
default_stiffness = [5000, 5000, 5000, 100, 100, 100]


# manipulation server
server = ms.ManipServer()
if not server.initialize(hardware_config_path):
    raise RuntimeError("Failed to initialize hardware server.")
server.set_high_level_maintain_position()

# self.image_transform = get_image_transform(
#     input_res=input_res, output_res=camera_res_hw, bgr_to_rgb=False
# )

if server.is_bimanual():
    id_list = [0, 1]
else:
    id_list = [0]

default_stiffness = np.diag(default_stiffness)

rgb_row_combined_buffer = []
rgb_buffer = []
output_rgb_buffer = []
for id in id_list:
    server.set_force_controlled_axis(Tr, n_af, id)
    server.set_stiffness_matrix(default_stiffness, id)
    # # initialize rgb buffers
    # # (c h) (n w)->n h w c
    # rgb_row_combined_buffer.append(
    #     np.zeros(
    #         (input_res[0] * 3, input_res[1] * query_sizes["sparse"]["rgb"]),
    #         dtype=np.uint8,
    #     )
    # )
    # rgb_buffer.append(
    #     np.zeros((query_sizes["sparse"]["rgb"], *input_res, 3), dtype=np.uint8)
    # )
    # output_rgb_buffer.append(
    #     np.zeros(
    #         (
    #             query_sizes["sparse"]["rgb"],
    #             camera_res_hw[0],
    #             camera_res_hw[1],
    #             3,
    #         ),
    #         dtype=np.uint8,
    #     )
    # )

print("[ManipServerEnv:] Waiting for hardware server to be ready...")
while not server.is_ready():
    time.sleep(0.1)

## Run Playback

In [ ]:
import PyriteUtility.spatial_math.spatial_utilities as su

# read data

ep = "episode_0"
data = buffer["data"][ep]

for key in data.keys():
    print("data keys: ", key)

print(data["ts_pose_fb_0"].shape)
print(data["gripper_0"].shape)

# get current pose
pose_W0 = np.squeeze(server.get_pose(1, 0))
SE3_W0 = su.pose7_to_SE3(pose_W0)

# read the first pose wp
pose_X0 = data["ts_pose_fb_0"][0]
SE3_0X = su.SE3_inv(su.pose7_to_SE3(pose_X0))

# read and convert all data to world
N = data["ts_pose_fb_0"].shape[0]
pose_WT = np.zeros((7, N))
for i in range(N):
    pose_XT = data["ts_pose_fb_0"][i]
    SE3_XT = su.pose7_to_SE3(pose_XT)
    SE3_0T = SE3_0X @ SE3_XT
    SE3_WT = SE3_W0 @ SE3_0T

    pose_WT[:, i] = su.SE3_to_pose7(SE3_WT)

# read all Gripper data

# start playing back
for i in range(N):
    pose_cmd = pose_WT[:, i]
    pose_fb = np.squeeze(server.get_pose(1, 0))
    print("pose_fb:", pose_fb)
    print("pose_cmd:", pose_cmd)

    input("Press Enter to execute...")
    timenow_ms = server.get_timestamp_now_ms()
    time_cmd = timenow_ms + 100
    server.schedule_waypoints(pose_cmd, time_cmd, 0)
    time.sleep(0.1)